In [2]:
import pandas as pd

path = "Anonymized MSI WN25 Graduates.xlsx - Report 1.csv"
df_raw = pd.read_csv(path, dtype=str)
print(f"Raw shape: {df_raw.shape}")
display(df_raw.head())
display(df_raw.columns.tolist())

Raw shape: (4584, 15)


,Emplid,Admit Term Descrshort,Acad Prog.Acad Career,Exp Grad Term Descrshort,Completion Term Descrshort,Acad Prog,Acad Prog Descr,Acad Plan,Acad Plan Descr,Term,Term Descrshort,Stdnt Car Term.Acad Career,Crse ID,Subject,Catalog Nbr
0,00604051,FA 2023,GINF,WN 2025,WN 2025,02003,Information Mas,9445MSI,User Experience Design MSI,2460,FA 2023,GINF,017281,SI,588
1,00604051,FA 2023,GINF,WN 2025,WN 2025,02003,Information Mas,9445MSI,User Experience Design MSI,2460,FA 2023,GINF,042104,SI,501
2,00604051,FA 2023,GINF,WN 2025,WN 2025,02003,Information Mas,9445MSI,User Experience Design MSI,2460,FA 2023,GINF,044772,SI,582
3,00604051,FA 2023,GINF,WN 2025,WN 2025,02003,Information Mas,9445MSI,User Experience Design MSI,2460,FA 2023,GINF,047246,SI,506
4,00604051,FA 2023,GINF,WN 2025,WN 2025,02003,Information Mas,9445MSI,User Experience Design MSI,2460,FA 2023,GINF,048129,SI,505


['Emplid',
 'Admit Term Descrshort',
 'Acad Prog.Acad Career',
 'Exp Grad Term Descrshort',
 'Completion Term Descrshort',
 'Acad Prog',
 'Acad Prog Descr',
 'Acad Plan',
 'Acad Plan Descr',
 'Term',
 'Term Descrshort',
 'Stdnt Car Term.Acad Career',
 'Crse ID',
 'Subject',
 'Catalog Nbr']

**Data Cleaning**

In [3]:
# Clean column names
df_raw.columns = (
    df_raw.columns.str.strip()
    .str.replace(r"[.\s]+", "_", regex=True)
    .str.replace("-", "", regex=False)
)

In [4]:
# Build derived columns (matching prior years' format)
df_raw["Subject"]     = df_raw["Subject"].str.strip()
df_raw["Catalog_Nbr"] = df_raw["Catalog_Nbr"].str.strip()
df_raw["Course_Code"] = df_raw["Subject"] + " " + df_raw["Catalog_Nbr"]
df_raw["Outside_UMSI"] = df_raw["Subject"].apply(lambda x: "No" if x == "SI" else "Yes")
df_raw["Emplid"]      = df_raw["Emplid"].str.strip()

In [5]:
# Remove the MSW student (not an MSI student)
# Emplid 46062495 is in "Int Prac IH MH&Subst Abuse MSW" — all SW courses, no SI 699
msw_mask = df_raw["Acad_Plan_Descr"].str.contains("MSW", na=False)
print(f"Dropping {msw_mask.sum()} rows from {df_raw.loc[msw_mask, 'Emplid'].nunique()} MSW student(s)")
df = df_raw[~msw_mask].copy()

Dropping 28 rows from 1 MSW student(s)


In [6]:
print(f"Cleaned shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
display(df[["Emplid", "Acad_Plan_Descr", "Course_Code", "Outside_UMSI"]].head(10))

Cleaned shape: (4556, 17)
Columns: ['Emplid', 'Admit_Term_Descrshort', 'Acad_Prog_Acad_Career', 'Exp_Grad_Term_Descrshort', 'Completion_Term_Descrshort', 'Acad_Prog', 'Acad_Prog_Descr', 'Acad_Plan', 'Acad_Plan_Descr', 'Term', 'Term_Descrshort', 'Stdnt_Car_Term_Acad_Career', 'Crse_ID', 'Subject', 'Catalog_Nbr', 'Course_Code', 'Outside_UMSI']


,Emplid,Acad_Plan_Descr,Course_Code,Outside_UMSI
0,00604051,User Experience Design MSI,SI 588,No
1,00604051,User Experience Design MSI,SI 501,No
2,00604051,User Experience Design MSI,SI 582,No
3,00604051,User Experience Design MSI,SI 506,No
4,00604051,User Experience Design MSI,SI 505,No
5,00604051,User Experience Design MSI,SI 622,No
6,00604051,User Experience Design MSI,SI 511,No
7,00604051,User Experience Design MSI,SI 520,No
8,00604051,User Experience Design MSI,SI 539,No
9,00604051,User Experience Design MSI,SI 681,No


**Data Exploration**

In [7]:
all_students = set(df["Emplid"].unique())
print(f"Number of Students: {len(all_students)}")
print(f"Number of Unique Courses: {df['Course_Code'].nunique()}")
print(f"\nMost Popular Courses:")
print(df["Course_Code"].value_counts().head(10))
print(f"\nAcad Plan distribution:")
print(df.groupby("Acad_Plan_Descr")["Emplid"].nunique().sort_values(ascending=False))


Number of Students: 269
Number of Unique Courses: 294

Most Popular Courses:
Course_Code
SI 699    257
SI 501    244
SI 539    187
SI 681    178
SI 506    177
SI 582    172
SI 588    162
SI 622    162
SI 520    148
SI 511    134
Name: count, dtype: int64

Acad Plan distribution:
Acad_Plan_Descr
User Experience Design MSI        133
Big Data Analytics MSI             59
Information MSI                    41
User-Centered Agile Dev MSI        26
Master's ThesisOption Prog MSI     10
Name: Emplid, dtype: int64


In [8]:
si699_students = set(df.loc[df["Course_Code"] == "SI 699", "Emplid"])
print("SI 699:", len(si699_students))

SI 699: 255


In [9]:
mtop_students = set(df.loc[df["Course_Code"].isin(["SI 697", "SI 698"]), "Emplid"])
print("MTOP:", mtop_students, len(mtop_students))

MTOP: {'90021333', '79370320', '39360886', '16524330', '70650357', '90567684', '14390965', '44364822', '38132522', '88530246'} 10


In [10]:
missing_students = all_students - (si699_students | mtop_students)
print("Missing Students:", missing_students, len(missing_students))

Missing Students: {'23505102', '01664949', '63795798', '02068614'} 4


Student: 02068614 -- BDA (haven't taken 699)

Student: 63795798 -- BDA (haven't taken 699, expected graduation date is WN 27, but isn't found in the "Anonymized MSI Graduates W26 + W27" dataset)

Student: 01664949 -- LAKES (haven't taken 699, expected graduation date is WN 27, but isn't found in the "Anonymized MSI Graduates W26 + W27" dataset)

Student: 23505102 -- UX (haven't taken 699, expected graduation date is WN 27, but isn't found in the "Anonymized MSI Graduates W26 + W27" dataset)

**Note: 255 Students who took 699 + 10 MTOP + 4 Students who have taken neither = 269 Students. Math tracks**

**Track Assignment & Filtering**

In [11]:
# Map tracks directly from Acad Plan Descr
plan_to_track = {
    "User Experience Design MSI":    "UX",
    "Big Data Analytics MSI":        "BDA",
    "User-Centered Agile Dev MSI":   "UCAD",
    "Information MSI":               "LAKES",
    "Master's ThesisOption Prog MSI": "MTOP",
}
df["Track"] = df["Acad_Plan_Descr"].map(plan_to_track)

In [12]:
# Exclude MTOP + students without SI 699
df_filtered = df[df["Track"] != "MTOP"].copy()
si699_ids = set(df_filtered.loc[df_filtered["Course_Code"] == "SI 699", "Emplid"])
df_filtered = df_filtered[df_filtered["Emplid"].isin(si699_ids)].copy()

print(f"Students for analysis: {df_filtered['Emplid'].nunique()}")
print(f"\nTrack distribution:")
print(df_filtered.groupby("Track")["Emplid"].nunique().sort_values(ascending=False))

Students for analysis: 255

Track distribution:
Track
UX       132
BDA       57
LAKES     40
UCAD      26
Name: Emplid, dtype: int64


**Elective Analysis Setup**

In [13]:
UNIVERSAL_EXCLUDE = {"SI 501","SI 505","SI 506","SI 681","SI 690","SI 699"}

REQUIRED = {
    "BDA": {"SI 504","SI 507","SI 544","SI 568","SI 602","SI 618",
            "SI 670","SI 671","SI 608","SI 630","SI 649","SI 650","SI 561"},
    "UX":  {"SI 520","SI 539","SI 582","SI 588","SI 622",
            "SI 529","SI 552","SI 559","SI 612","SI 616","SI 658","SI 659","SI 684"},
    "UCAD":{"SI 504","SI 539","SI 579","SI 582","SI 588","SI 622","SI 664","SI 669"},
    "LAKES":{"SI 510","SI 580","SI 647","SI 547","SI 623","SI 538","SI 626","SI 632",
             "SI 633","SI 643","SI 583","SI 585","SI 666","SI 667","SI 504","SI 676",
             "SI 539","SI 564","SI 639","SI 622"},
}

**SI Elective Analysis**

In [14]:
results = {}
for track in ["BDA", "UX", "UCAD", "LAKES"]:
    exclude = UNIVERSAL_EXCLUDE | REQUIRED[track]
    track_ids = df_filtered[df_filtered["Track"] == track]
    n_students = track_ids["Emplid"].nunique()

    track_df = track_ids[~track_ids["Course_Code"].isin(exclude)]

    counts = (
        track_df.groupby("Course_Code")["Emplid"]
        .nunique().sort_values(ascending=False)
        .rename("Student_Count").reset_index()
    )
    counts["Pct_of_Track"] = (counts["Student_Count"] / n_students * 100).round(1)
    results[track] = counts

    print(f"\n\u2500\u2500 {track} (n={n_students} students) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
    print(counts.head(11).to_string(index=False))


── BDA (n=57 students) ──────────────────
Course_Code  Student_Count  Pct_of_Track
     SI 564             41          71.9
     SI 664             33          57.9
     SI 539             27          47.4
     SI 644             24          42.1
     SI 563             15          26.3
     SI 511              8          14.0
     SI 579              7          12.3
   ENTR 500              7          12.3
     SI 582              6          10.5
     SI 588              5           8.8
     SI 669              5           8.8

── UX (n=132 students) ──────────────────
Course_Code  Student_Count  Pct_of_Track
     SI 511             77          58.3
     SI 611             45          34.1
     SI 594             34          25.8
     SI 504             24          18.2
   ENTR 500             21          15.9
     SI 538             21          15.9
     SI 682             17          12.9
     SI 579             16          12.1
     SI 631             15          11.4
     SI 564 

**Outside UMSI Elective Analysis**

In [15]:
outside_results = {}
for track in ["BDA", "UX", "UCAD", "LAKES"]:
    exclude = UNIVERSAL_EXCLUDE | REQUIRED[track]
    track_ids = df_filtered[df_filtered["Track"] == track]
    n_students = track_ids["Emplid"].nunique()

    outside_df = track_ids[
        (~track_ids["Course_Code"].isin(exclude)) &
        (track_ids["Outside_UMSI"] == "Yes")
    ]

    counts = (
        outside_df.groupby("Course_Code")["Emplid"]
        .nunique().sort_values(ascending=False)
        .rename("Student_Count").reset_index()
    )
    counts["Pct_of_Track"] = (counts["Student_Count"] / n_students * 100).round(1)
    outside_results[track] = counts

    print(f"\n\u2500\u2500 {track} (n={n_students} students) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
    print(counts.head(12).to_string(index=False))


── BDA (n=57 students) ──────────────────
Course_Code  Student_Count  Pct_of_Track
   ENTR 500              7          12.3
   ENTR 560              3           5.3
     TO 618              3           5.3
     TO 512              3           5.3
WRITING 993              2           3.5
  STATS 503              2           3.5
   EECS 403              2           3.5
    ELI 536              2           3.5
  STATS 513              2           3.5
     TO 638              2           3.5
 PUBPOL 564              1           1.8
 PUBPOL 577              1           1.8

── UX (n=132 students) ──────────────────
 Course_Code  Student_Count  Pct_of_Track
    ENTR 500             21          15.9
    ENTR 560             12           9.1
     IOE 548             10           7.6
      TO 548             10           7.6
    ENTR 550              9           6.8
   UARTS 560              8           6.1
    ENTR 599              8           6.1
     MKT 625              7           5.3
   

In [16]:
# Export full elective list for all tracks — for student-facing course directory
import json

full_course_map = {}

for track, df_result in results.items():
    # Only include SI courses (not outside UMSI)
    si_only = df_result[df_result['Course_Code'].str.startswith('SI ')]
    for _, row in si_only.iterrows():
        code = row['Course_Code']
        name = row.get('Crse_Descr') or row.get('Crse Descr') or ''
        if code not in full_course_map:
            full_course_map[code] = {'name': name, 'tracks': []}
        if track not in full_course_map[code]['tracks']:
            full_course_map[code]['tracks'].append(track)

# Print as JavaScript array ready to paste into your HTML file
print('const allCourses = [')
for code in sorted(full_course_map.keys(), key=lambda x: int(x.split()[1])):
    info = full_course_map[code]
    tracks = sorted([t.lower() for t in info['tracks']])
    tracks_str = ', '.join(f"'{t}'" for t in tracks)
    name = info['name'].replace("'", "\\'")
    print(f"  {{ code: '{code}', name: '{name}', tracks: [{tracks_str}] }},")
print('];')

const allCourses = [
  { code: 'SI 311', name: '', tracks: ['ux'] },
  { code: 'SI 500', name: '', tracks: ['bda'] },
  { code: 'SI 504', name: '', tracks: ['ux'] },
  { code: 'SI 507', name: '', tracks: ['lakes', 'ucad', 'ux'] },
  { code: 'SI 510', name: '', tracks: ['bda', 'ux'] },
  { code: 'SI 511', name: '', tracks: ['bda', 'lakes', 'ucad', 'ux'] },
  { code: 'SI 512', name: '', tracks: ['bda'] },
  { code: 'SI 515', name: '', tracks: ['ux'] },
  { code: 'SI 519', name: '', tracks: ['bda', 'lakes', 'ucad', 'ux'] },
  { code: 'SI 520', name: '', tracks: ['bda', 'lakes', 'ucad'] },
  { code: 'SI 529', name: '', tracks: ['bda', 'lakes', 'ucad'] },
  { code: 'SI 534', name: '', tracks: ['bda', 'lakes', 'ux'] },
  { code: 'SI 538', name: '', tracks: ['ucad', 'ux'] },
  { code: 'SI 539', name: '', tracks: ['bda'] },
  { code: 'SI 540', name: '', tracks: ['bda', 'lakes', 'ucad', 'ux'] },
  { code: 'SI 542', name: '', tracks: ['bda', 'ux'] },
  { code: 'SI 544', name: '', tracks: ['ucad'